Importing Dependencies

In [30]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [20]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

Importing Dataset

In [4]:
dataset =  pd.read_csv('/content/WELFake_Dataset.csv', engine='python', on_bad_lines='skip')
dataset.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [52]:
dataset.tail()

,Unnamed: 0,title,text,label
9159,9139,Franklin Graham: The media didn’t understand t...,X close We have experienced tremendous growth ...,1
9160,9140,WATCH: Jailed Palestinian Terrorist And Hunger...,TEL AVIV — The Israeli Prison Service relea...,0
9161,9141,Vladimir Putin at the Valdai International Dis...,Vladimir Putin at the Valdai International Dis...,1
9162,9142,"EUROPE CRASHES AND BURNS, As EU Officials FINA...",The Genie is out of the bottle. Europe will ne...,1
9163,9143,Harley-Davidson to Pay $12 Million Fine in Pol...,WASHINGTON — has agreed to pay a $12 mill...,0


In [5]:
dataset['text'][0]

'No comment is expected from Barack Obama Members of the #FYF911 or #FukYoFlag and #BlackLivesMatter movements called for the lynching and hanging of white people and cops. They encouraged others on a radio show Tuesday night to  turn the tide  and kill white people and cops to send a message about the killing of black people in America.One of the F***YoFlag organizers is called  Sunshine.  She has a radio blog show hosted from Texas called,  Sunshine s F***ing Opinion Radio Show. A snapshot of her #FYF911 @LOLatWhiteFear Twitter page at 9:53 p.m. shows that she was urging supporters to  Call now!! #fyf911 tonight we continue to dismantle the illusion of white Below is a SNAPSHOT Twitter Radio Call Invite   #FYF911The radio show aired at 10:00 p.m. eastern standard time.During the show, callers clearly call for  lynching  and  killing  of white people.A 2:39 minute clip from the radio show can be heard here. It was provided to Breitbart Texas by someone who would like to be referred to

In [6]:
dataset.isnull().sum()

,0
Unnamed: 0,0
title,66
text,26
label,20


In [7]:
dataset.shape

(9164, 4)

Removing/ Handling Missing Values

There are **9164** datapoints in that nearly 112 datapoint are missing /  null values. so droping those null values will not make an big effect in this scenario. So I am going to choose Droping for handing those missing value

In [8]:
dataset_without_missing_values = dataset.dropna(axis=0)

In [9]:
dataset_without_missing_values.shape

(9077, 4)

In [10]:
dataset_without_missing_values.isnull().sum()

,0
Unnamed: 0,0
title,0
text,0
label,0


Checking for data imbalance

In [32]:
dataset_without_missing_values['label'].value_counts()

label
1                                                  4728
0                                                  4347
 is just unbelievably rich from North Sea oil.        1
 за участие. Благодарю вас.                           1
Name: count, dtype: int64


in this case the dataset is balanced

Feature Selection

1. in this case I am just going to use title and label column due to lack of computation.

In [18]:
X = dataset_without_missing_values['title'].values
Y = dataset_without_missing_values['label'].values

In [19]:
print(X)
print(Y)

['LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO]'
 'UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MOST CHARLOTTE RIOTERS WERE “PEACEFUL” PROTESTERS…In Her Home State Of North Carolina [VIDEO]'
 'Bobby Jindal, raised Hindu, uses story of Christian conversion to woo evangelicals for potential 2016 bid'
 ...
 'Vladimir Putin at the Valdai International Discussion Club : «Shaping the World of Tomorrow», by Vladimir Putin'
 'EUROPE CRASHES AND BURNS, As EU Officials FINALLY Admit Secret About “Refugees” They’ve Known For A While'
 'Harley-Davidson to Pay $12 Million Fine in Pollution Settlement - The New York Times']
['1' '1' '0' ... '1' '1' '0']


Stemming:

the process of converting the words to it's root word

In [21]:
porter_stemmer = PorterStemmer()

In [22]:
def stemming(content):
  stem_content = re.sub('[^a-zA-Z]'," ",content)
  stem_content = stem_content.lower()
  stem_content = stem_content.split()
  stem_content = [porter_stemmer.stem(word) for word in stem_content if not word in stopwords.words('english')]
  stem_content = ' '.join(stem_content)
  return stem_content


In [23]:
X = dataset_without_missing_values['title'].apply(stemming)

In [24]:
print(X)

0       law enforc high alert follow threat cop white ...
2       unbeliev obama attorney gener say charlott rio...
3       bobbi jindal rais hindu use stori christian co...
4       satan russia unv imag terrifi new supernuk wes...
5       time christian group sue amazon splc design ha...
                              ...                        
9159    franklin graham media understand god factor tr...
9160    watch jail palestinian terrorist hunger strike...
9161    vladimir putin valdai intern discuss club shap...
9162    europ crash burn eu offici final admit secret ...
9163    harley davidson pay million fine pollut settle...
Name: title, Length: 9077, dtype: object


Vectorizing the data:

converting the text data into numerical data using the TF-IDF Vectorizer.

In [25]:
vectorizer  =TfidfVectorizer()

In [27]:
vectorized_X = vectorizer.fit_transform(X)

In [29]:
print(vectorized_X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 79831 stored elements and shape (9077, 9087)>
  Coords	Values
  (0, 4526)	0.2409773425869288
  (0, 2600)	0.3236941802442232
  (0, 3692)	0.27529983170756966
  (0, 175)	0.3289723790530952
  (0, 3044)	0.2979104472356995
  (0, 8103)	0.26219708060239794
  (0, 1721)	0.2615703166094318
  (0, 8868)	0.2011916003769295
  (0, 805)	0.3457634772576234
  (0, 3191)	0.42215817450528237
  (0, 8054)	0.26479565749043754
  (0, 8664)	0.13732258933984892
  (1, 8664)	0.13500274779253868
  (1, 8414)	0.3487630851905186
  (1, 5528)	0.17116131855849706
  (1, 465)	0.29020686581755323
  (1, 3255)	0.26744361381929543
  (1, 6992)	0.162869069355756
  (1, 1330)	0.35395211742489874
  (1, 6769)	0.36662619683613584
  (1, 5876)	0.2819591014334065
  (1, 6281)	0.2314605043535776
  (1, 3754)	0.26901720445515875
  (1, 7650)	0.2037438509657077
  (1, 5476)	0.2203507024285866
  :	:
  (9074, 8591)	0.3178575125916665
  (9074, 1495)	0.2832747001633116
  (9074, 7213)	0.29

Splitting the data into train and test data

In [33]:
X_train , X_test , Y_train , Y_test = train_test_split(vectorized_X,Y,test_size=0.2,random_state=2)

In [34]:
print(X_train.shape , Y_train.shape)

(7261, 9087) (7261,)


In [35]:
print(X_test.shape ,Y_test.shape)

(1816, 9087) (1816,)


Loading and Trainig the Model

In [36]:
model = LogisticRegression()

In [38]:
model.fit(X_train,Y_train)

LogisticRegression()

In [39]:
X_train_perdiction = model.predict(X_train)
X_train_accuracy = accuracy_score(X_train_perdiction,Y_train)

In [40]:
print(X_train_accuracy)

0.9500068861038424


In [41]:
X_test_perdiction = model.predict(X_test)
X_test_accuracy = accuracy_score(X_test_perdiction,Y_test)

In [42]:
print(X_test_accuracy)

0.861784140969163


Building Prediction Model

In [67]:
dataset['title'][3]

'Bobby Jindal, raised Hindu, uses story of Christian conversion to woo evangelicals for potential 2016 bid'

In [68]:
input = 'Bobby Jindal, raised Hindu, uses story of Christian conversion to woo evangelicals for potential 2016 bid'
stemmed_input = stemming(input)

vectorized_input = vectorizer.transform([stemmed_input])

prediction = model.predict(vectorized_input)

print(prediction)
if prediction[0] == '0':
  print("Real News")
else:
  print("Fake News")

['0']
Real News
